# PulmoGuard — Training & Evaluation (Google Colab, Free T4)

**Selective-Prediction Chest X-Ray Triage System**

This notebook trains an EfficientNet-B0 pneumonia classifier and evaluates it
with Monte Carlo Dropout uncertainty estimation, producing the project's
headline artifact: a **risk-coverage curve**.

**Before you start:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

**Workflow:**
1. Mount Google Drive (checkpoints/plots persist across session disconnects)
2. Clone this repo
3. Install dependencies
4. Download the dataset from Kaggle
5. Train (`src/pulmoguard/train.py`)
6. Evaluate + plot risk-coverage curve (`src/pulmoguard/evaluate.py`)
7. Download the trained checkpoint to use locally (see repo README)


In [ ]:
# 1. Mount Google Drive
# This ensures checkpoints and plots survive a Colab disconnect.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/pulmoguard-cv'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Drive project directory: {DRIVE_PROJECT_DIR}")


In [ ]:
# 2. Clone the repository
# If you've pushed this project to GitHub, clone it directly (recommended).
# Otherwise, upload the provided pulmoguard-cv.zip via the Colab file browser
# and unzip it instead (see commented alternative below).

REPO_URL = "https://github.com/<your-username>/pulmoguard-cv.git"  # <-- update this

%cd /content
!git clone $REPO_URL
%cd pulmoguard-cv

# --- Alternative if you uploaded a zip instead of using GitHub: ---
# from google.colab import files
# uploaded = files.upload()  # select pulmoguard-cv.zip
# !unzip -q pulmoguard-cv.zip -d /content
# %cd /content/pulmoguard-cv


In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .


In [ ]:
# 4. Verify GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")


## Download the dataset

You need a free Kaggle account and API token:
1. Go to kaggle.com → Account → Create New API Token → downloads `kaggle.json`
2. Upload it below when prompted


In [ ]:
# 5. Kaggle API setup + dataset download
from google.colab import files

print("Upload your kaggle.json API token file:")
uploaded = files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!bash scripts/download_data.sh data


In [ ]:
# Sanity-check dataset layout and class balance
import os

data_root = "data/chest_xray"
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        path = os.path.join(data_root, split, cls)
        n = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"{split:6s} / {cls:10s}: {n} images")


## Train

Trains EfficientNet-B0 (ImageNet-pretrained) with mixed precision, saving the
best checkpoint (by validation macro-F1) to `outputs/checkpoints/`.
Expected time on a free-tier T4: roughly 30–50 minutes for the default
8-epoch config with early stopping.


In [ ]:
# 6. Train the model
!python -m pulmoguard.train --config configs/config.yaml --data-root data/chest_xray --output-dir outputs


In [ ]:
# Copy checkpoint to Drive immediately so it survives a disconnect
import shutil

os.makedirs(f"{DRIVE_PROJECT_DIR}/checkpoints", exist_ok=True)
shutil.copy(
    "outputs/checkpoints/pulmoguard_best.pt",
    f"{DRIVE_PROJECT_DIR}/checkpoints/pulmoguard_best.pt",
)
print("Checkpoint backed up to Google Drive.")


## Evaluate: Risk-Coverage Curve + Calibration

This is the project's headline result. Instead of a single forced-choice
accuracy number, we measure accuracy as a function of how much of the test
set the model is willing to answer, using MC-Dropout predictive entropy to
decide what to abstain on.


In [ ]:
# 7. Evaluate
!python -m pulmoguard.evaluate \
    --config configs/config.yaml \
    --checkpoint outputs/checkpoints/pulmoguard_best.pt \
    --data-root data/chest_xray \
    --output-dir outputs


In [ ]:
# Display the risk-coverage curve and confusion matrix inline
from IPython.display import Image, display

display(Image(filename="outputs/plots/risk_coverage_curve.png"))
display(Image(filename="outputs/plots/confusion_matrix.png"))


In [ ]:
# Print the headline metrics
import json

with open("outputs/metrics/evaluation_results.json") as f:
    results = json.load(f)

print(f"Forced (100% coverage) accuracy: {results['forced_accuracy_100pct_coverage']:.4f}")
print(f"Expected Calibration Error:      {results['expected_calibration_error']:.4f}")
print()
print("Coverage -> Accuracy:")
for cov, acc in zip(results['risk_coverage_curve']['coverage'], results['risk_coverage_curve']['accuracy']):
    print(f"  {cov:5.2f}  ->  {acc:.4f}")


In [ ]:
# Back up plots and metrics to Drive as well
os.makedirs(f"{DRIVE_PROJECT_DIR}/plots", exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_DIR}/metrics", exist_ok=True)

for fname in ["risk_coverage_curve.png", "confusion_matrix.png"]:
    shutil.copy(f"outputs/plots/{fname}", f"{DRIVE_PROJECT_DIR}/plots/{fname}")

shutil.copy(
    "outputs/metrics/evaluation_results.json",
    f"{DRIVE_PROJECT_DIR}/metrics/evaluation_results.json",
)
print("Plots and metrics backed up to Google Drive.")


## Next steps: run locally

Download `pulmoguard_best.pt` from Google Drive (or directly from the Colab
file browser under `outputs/checkpoints/`) and place it at:

```
outputs/checkpoints/pulmoguard_best.pt
```

in your local clone of this repository. Then follow the "Run inference
locally" section of the README to serve predictions via the CLI or the
FastAPI app — no GPU required for inference.


In [ ]:
# Optional: download the checkpoint directly from this Colab session
from google.colab import files
files.download("outputs/checkpoints/pulmoguard_best.pt")
